# SNNAI Phase0.5 — Bee Navigation / Path Integration Reproduction

This notebook reproduces a minimal path-integration model inspired by honeybee navigation.

**Goal**: integrate velocity signals over time to estimate position, and compare with true position.

In [ ]:
# Install dependencies
!pip install -q snntorch==0.9.4

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import snntorch as snn

print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())

## 1. Generate a 2D random walk trajectory

In [ ]:
np.random.seed(42)
torch.manual_seed(42)

T = 500
dt = 0.01  # 10 ms

# Random heading changes
heading = np.cumsum(np.random.randn(T) * 0.1)
speed = np.clip(1.0 + 0.3 * np.random.randn(T), 0.2, 1.5)

vx = speed * np.cos(heading)
vy = speed * np.sin(heading)

true_x = np.cumsum(vx) * dt
true_y = np.cumsum(vy) * dt

print('True final position:', true_x[-1], true_y[-1])

## 2. Encode velocity as spike trains

In [ ]:
def encode_velocity_signed(v, max_rate=0.3, n_neurons=20):
    """Encode signed velocity into positive/negative spike populations."""
    v_max = np.abs(v).max() + 1e-8
    v_pos = np.maximum(v, 0) / v_max
    v_neg = np.maximum(-v, 0) / v_max
    half = n_neurons // 2
    rates_pos = v_pos[:, None] * np.ones((1, half)) * max_rate
    rates_neg = v_neg[:, None] * np.ones((1, half)) * max_rate
    spikes_pos = (np.random.rand(T, half) < rates_pos).astype(float)
    spikes_neg = (np.random.rand(T, half) < rates_neg).astype(float)
    return torch.tensor(np.concatenate([spikes_pos, spikes_neg], axis=1), dtype=torch.float32)

n_vel = 20  # 10 positive + 10 negative per axis
spk_vx = encode_velocity_signed(vx, n_neurons=n_vel)
spk_vy = encode_velocity_signed(vy, n_neurons=n_vel)
spk_input = torch.cat([spk_vx, spk_vy], dim=1)
print('Input spikes shape:', spk_input.shape)

## 3. Build two integrator neurons (x and y)

In [ ]:
n_input = 2 * n_vel
half = n_vel // 2

W = nn.Linear(n_input, 2, bias=False)

# Scale weights so that integrated input approximates true displacement
# Positive half contributes +, negative half contributes -
with torch.no_grad():
    W.weight.zero_()
    # x-integrator: + from vx positive, - from vx negative
    W.weight[0, :half] = 0.02
    W.weight[0, half:n_vel] = -0.02
    # y-integrator: + from vy positive, - from vy negative
    W.weight[1, n_vel:n_vel+half] = 0.02
    W.weight[1, n_vel+half:2*n_vel] = -0.02

beta = 0.999
lif = snn.Leaky(beta=beta, threshold=1e6, learn_threshold=False)
print('Integrator beta:', beta)

## 4. Run path integration

In [ ]:
estimates = []
mem = lif.init_leaky()
if mem.dim() == 0:
    mem = torch.zeros(2)

for t in range(T):
    current = W(spk_input[t])
    _, mem = lif(current, mem)
    estimates.append(mem.detach().numpy())

estimates = np.array(estimates)
est_x, est_y = estimates[:, 0], estimates[:, 1]

print('Estimated final position:', est_x[-1], est_y[-1])
print('True final position:', true_x[-1], true_y[-1])

final_error = np.sqrt((est_x[-1] - true_x[-1])**2 + (est_y[-1] - true_y[-1])**2)
trajectory_length = np.sqrt(true_x[-1]**2 + true_y[-1]**2)
relative_error = final_error / (trajectory_length + 1e-8)
print(f'Final error: {final_error:.4f}')
print(f'Relative error: {relative_error:.4f}')